# Spectral Decomposition Approach


Simplifed noteebook demonstrating night sky decomposition approach.

Sky spectrum decomposed into model of airglow sky lines and continuum components:

I. Continuum components include:

- Moon (high-resolution solar spectrum rebinned to LVM sampling and convolved to LVM Gaussian LSF and multiplied by B-spline multiplicative continuum). This mimicks Moon itself and Zodi components.

- Diffuse components (interpolated from low-resolution PALACE diffuse continuum components)

  - Hydroperoxyl ($HO_2$): This is the predominant continuum component in the near-infrared range, characterized by a prominent emission peak at 1.51 µm.

  - Iron Monoxide ($FeO$) and other molecules: This component dominates the visual wavelength range (roughly 500 to 720 nm) and includes the $FeO$ "orange arc" bands, with potential additional contributions from $NiO$ or $OFeOH$.

  - Unresolved Molecular Oxygen ($O_2$): Located in the ultraviolet (UVB) range, this component accounts for weak, unresolved bands from high-energy electronic states (specifically $c^1\Sigma^-_u$, $A'^3\Delta_u$, and $A^3\Sigma^+_u$).


II. Airglow sky components include:

  - Atomic Oxygen (O I) emission lines within the visual wavelength range.

  - Sodium (Na I): doublet at 5889.95 and 5895.92 Å, formed in the mesospheric Na layer at about 92 km by chemiluminescent reactions of meteoric sodium; typical D2 / D1 ≈ 1.7.
  
  - Potassium (K I): doublet at 7664.90 and 7698.96 Å, formed in the mesospheric K layer at about 89 km by chemistry similar to Na; typical D2 / D1 ≈ 1.67.
  
  - Nitrogen (N I): [ N I ] [NI] doublet at 5197.90 and 5200.26 Å, formed higher in the ionosphere at about 250 km via dissociative recombination; typical 5198 / 5200 ≈ 1.76.


  - Unresolved Molecular Oxygen ($O_2$): Located in the ultraviolet (UVB) range, this component accounts for weak, unresolved bands from high-energy electronic states (specifically $c^1\Sigma^-_u$, $A'^3\Delta_u$, and $A^3\Sigma^+_u$).

  - Hydroxyl (OH): This component accounts for the hydroxyl emission lines in the visual wavelength range (roughly 500 to 720 nm).

# Manual in-place experiments -- ND

## Imports and helpers

Load the core packages and define the helper functions used throughout the notebook.

In [3]:
import plotly.graph_objects as go
from astropy.io import fits
from astropy.table import Table
import numpy as np

FACTOR = 1e14
LSF_SIGMA = 0.5
T_O2 = 191.5 # in K

PALACE_DIR = '../'
MEDIAN_STACK_DIR = '../'

f = MEDIAN_STACK_DIR+'lvmsframe_median_stack_1.2.1_limit100.fits'
fits.info(f)

wave = fits.getdata(f, "WAVE").astype(np.float64)
flx_sky1 = fits.getdata(f, "FLUX_SKY_NEAR").astype(np.float64) * FACTOR
flx_sky2 = fits.getdata(f, "FLUX_SKY_FAR").astype(np.float64) * FACTOR
flx_sci = fits.getdata(f, "FLUX_SCI").astype(np.float64) * FACTOR
meta = Table(fits.getdata(f, "META"))
flx_ivar = 1.0 + np.zeros_like(flx_sci)  # fits.getdata(f, "FLUX_IVAR").astype(np.float64)
# flx_err = 1.0 / np.sqrt(flx_ivar)
# flx_sci = flx_flux + flx_sky
flx_sci.shape


def plot_fit_result(result, idx):
    bestfit = result.bestfit
    bestfit_lsf = result.bestfit_lsf
    comp_Moon = result.components["moon"]
    comp_DIFFUSE = result.components["diffuse"]

    resid = flx_sci[idx] - bestfit
    resid_lsf = flx_sci[idx] - bestfit_lsf
    resid_level = -3.0 * np.nanstd(resid)

    err_plot = 1.0 / np.sqrt(np.where(flx_ivar[idx] > 0, flx_ivar[idx], np.nan))

    fig = go.Figure()
    fig.add_trace(go.Scattergl(x=wave, y=flx_sci[idx], mode="lines", name="Observed",
                            line=dict(color="black", width=1)))
    fig.add_trace(go.Scattergl(x=wave, y=bestfit_lsf, mode="lines", name="Best-fit + LSF",
                            line=dict(color="darkorange", width=1.5)))
    fig.add_trace(go.Scattergl(x=wave, y=comp_Moon, mode="lines", name="Moon",
                            line=dict(color="#4e79a7", width=1, dash="dash")))
    fig.add_trace(go.Scattergl(x=wave, y=comp_DIFFUSE, mode="lines", name="Diffuse",
                            line=dict(color="#59a14f", width=1, dash="dash")))
    fig.add_trace(go.Scattergl(x=wave, y=resid_level + resid_lsf, mode="lines", name="Residual + LSF",
                            line=dict(color="seagreen", width=1)))
    fig.add_trace(go.Scattergl(x=wave, y=resid_level + err_plot, mode="lines", name="+1sigma",
                            line=dict(color="gray", width=1, dash="dash")))
    fig.add_trace(go.Scattergl(x=wave, y=resid_level - err_plot, mode="lines", name="-1sigma",
                            line=dict(color="gray", width=1, dash="dash")))

    fig.update_layout(
        title={
            "text": (
                f"idx={idx} | T_O2={result.t_o2:.1f}±{result.t_o2_err:.1f} K<br>"
                f"{result.fit_summary}"
            ),
            "font": {"size": 14},
        },
        xaxis_title="λ (Å)",
        yaxis_title="Flux",
        template="plotly_white",
        height=520,
    )
    fig.show()


Filename: ../lvmsframe_median_stack_1.2.1_limit100.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU      12   ()      
  1  WAVE          1 ImageHDU         9   (12401,)   float32   
  2  FLUX_SCI      1 ImageHDU        10   (12401, 100)   float32   
  3  FLUX_SKY_NEAR    1 ImageHDU        10   (12401, 100)   float32   
  4  FLUX_SKY_FAR    1 ImageHDU        10   (12401, 100)   float32   
  5  FLUX_SCI_NOSKY    1 ImageHDU        10   (12401, 100)   float32   
  6  LSF_SCI       1 ImageHDU        10   (12401, 100)   float32   
  7  LSF_SKY_NEAR    1 ImageHDU        10   (12401, 100)   float32   
  8  LSF_SKY_FAR    1 ImageHDU        10   (12401, 100)   float32   
  9  META          1 BinTableHDU    107   100R x 48C   [512A, K, K, K, K, 32A, 32A, D, D, D, D, D, D, 8A, 8A, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, K, K, K, K, K, K, K, 16A, 512A]   


In [16]:
%load_ext autoreload
%autoreload 2
from sky_decomp.fit import SkyDecomp

decomposer = SkyDecomp(wave, lsf_sigma=LSF_SIGMA, base_dir=PALACE_DIR)
len(flx_sci)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


100

In [22]:
%%writefile sky_decomp_worker.py

def fit_chunk_worker(args):
    """Worker function: builds its own SkyDecomp and fits a chunk of spectra."""
    chunk_idxs, flx_sci_chunk, flx_sky1_chunk, flx_sky2_chunk, flx_ivar_chunk, wave, lsf_sigma, base_dir = args

    from sky_decomp.fit import SkyDecomp
    local_decomposer = SkyDecomp(wave, lsf_sigma=lsf_sigma, base_dir=base_dir)

    local_sci, local_sky1, local_sky2 = [], [], []
    for i in range(len(chunk_idxs)):
        local_sci.append(local_decomposer.fit(flx_sci_chunk[i],  flx_ivar_chunk[i], verbose=False, n_lsf_refits=3))
        local_sky1.append(local_decomposer.fit(flx_sky1_chunk[i], flx_ivar_chunk[i], verbose=False, n_lsf_refits=3))
        local_sky2.append(local_decomposer.fit(flx_sky2_chunk[i], flx_ivar_chunk[i], verbose=False, n_lsf_refits=3))

    return chunk_idxs, local_sci, local_sky1, local_sky2

Writing sky_decomp_worker.py


In [23]:
from tqdm.notebook import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
from sky_decomp_worker import fit_chunk_worker
import numpy as np

N_WORKERS = 4

idxs = list(range(len(flx_sci)))
chunks = np.array_split(idxs, N_WORKERS)

worker_args = [
    (
        list(chunk),
        flx_sci[chunk],
        flx_sky1[chunk],
        flx_sky2[chunk],
        flx_ivar[chunk],
        wave,
        LSF_SIGMA,
        PALACE_DIR,
    )
    for chunk in chunks
]

result_sci  = [None] * len(idxs)
result_sky1 = [None] * len(idxs)
result_sky2 = [None] * len(idxs)

with ProcessPoolExecutor(max_workers=N_WORKERS) as executor:
    futures = {executor.submit(fit_chunk_worker, args): tid
               for tid, args in enumerate(worker_args)}
    for future in tqdm(as_completed(futures), total=N_WORKERS, desc="Chunks done"):
        chunk_idxs, sci, sky1, sky2 = future.result()
        for i, idx in enumerate(chunk_idxs):
            result_sci[idx]  = sci[i]
            result_sky1[idx] = sky1[i]
            result_sky2[idx] = sky2[i]

Chunks done:   0%|          | 0/4 [00:00<?, ?it/s]

In [19]:
def results_to_fits(results, filename):
    """Write a list of SkyDecompResult objects to a FITS file.
    
    Scalar quantities go into a binary table (extension META).
    Spectral/coefficient arrays go into separate image extensions.
    
    Extensions:
        META         - BinTable with scalar fields per result
        COEF         - (n_results, n_coef) fit coefficients
        BESTFIT      - (n_results, n_wave) initial best-fit spectra
        BESTFIT_LSF  - (n_results, n_wave) LSF-refined best-fit spectra
        RESID        - (n_results, n_wave) residuals
        COMP_<KEY>   - (n_results, n_wave) per component spectra
    """
    rows = {
        "t_o2":               [r.t_o2 for r in results],
        "t_o2_err":           [r.t_o2_err for r in results],
        "reduced_chi2":       [r.reduced_chi2 for r in results],
        "r2":                 [r.r2 for r in results],
        "rms_resid":          [r.rms_resid for r in results],
        "resid_level":        [r.resid_level for r in results],
        "fit_status":         [r.fit_status for r in results],
        "fit_summary":        [r.fit_summary for r in results],
        "fit_elapsed_sec":    [r.fit_elapsed_sec for r in results],
        "peak_memory_mb":     [r.peak_memory_mb for r in results],
        "o2_fit_status":      [r.o2_fit_status for r in results],
        "o2_fit_summary":     [r.o2_fit_summary for r in results],
        "o2_fit_elapsed_sec": [r.o2_fit_elapsed_sec for r in results],
        "o2_valid_frac":      [r.o2_valid_frac for r in results],
    }
    t = Table(rows)

    def stack(attr):
        return np.vstack([getattr(r, attr) for r in results])

    # COEF header: store design_names as FITS keywords for reference
    coef_arr = stack("coef")
    coef_hdu = fits.ImageHDU(coef_arr, name="COEF")
    design_names = results[0].design_names
    for i, name in enumerate(design_names):
        coef_hdu.header[f"COEF{i:04d}"] = name

    hdul = fits.HDUList([
        fits.PrimaryHDU(),
        fits.BinTableHDU(t, name="META"),
        coef_hdu,
        fits.ImageHDU(stack("bestfit"),     name="BESTFIT"),
        fits.ImageHDU(stack("bestfit_lsf"), name="BESTFIT_LSF"),
        fits.ImageHDU(stack("resid"),       name="RESID"),
    ])

    comp_keys = list(results[0].components.keys())
    for key in comp_keys:
        arr = np.vstack([r.components[key] for r in results])
        hdul.append(fits.ImageHDU(arr, name=f"COMP_{key.upper()}"))

    hdul.writeto(filename, overwrite=True)
    print(f"Wrote {len(results)} results, {coef_arr.shape[1]} coefs, {len(comp_keys)} components → {filename}")

In [24]:
results_to_fits(result_sci, "sky_decomp_results_sci.fits")
results_to_fits(result_sky1, "sky_decomp_results_sky1.fits")
results_to_fits(result_sky2, "sky_decomp_results_sky2.fits")

Wrote 100 results, 442 coefs, 9 components → sky_decomp_results_sci.fits
Wrote 100 results, 442 coefs, 9 components → sky_decomp_results_sky1.fits
Wrote 100 results, 442 coefs, 9 components → sky_decomp_results_sky2.fits


In [8]:
hdul = fits.open("lvmsframe_median_stack_1.2.1_limit100_decomp_sci.fits")
hdul.info()

Filename: lvmsframe_median_stack_1.2.1_limit100_decomp_sci.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       4   ()      
  1  META          1 BinTableHDU     37   100R x 14C   [D, D, D, D, D, D, 6A, 114A, D, D, 1A, 112A, D, D]   
  2  COEF          1 BinTableHDU    893   100R x 442C   [D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, 

In [9]:
t = Table(hdul["COEF"].data)

In [10]:
t.colnames

['OH_000',
 'OH_001',
 'OH_002',
 'OH_003',
 'OH_004',
 'OH_005',
 'OH_006',
 'OH_007',
 'OH_008',
 'OH_009',
 'OH_010',
 'OH_011',
 'OH_012',
 'OH_013',
 'OH_014',
 'OH_015',
 'OH_016',
 'OH_017',
 'OH_018',
 'OH_019',
 'OH_020',
 'OH_021',
 'OH_022',
 'OH_023',
 'OH_024',
 'OH_025',
 'OH_026',
 'OH_027',
 'OH_028',
 'OH_029',
 'OH_030',
 'OH_031',
 'OH_032',
 'OH_033',
 'OH_034',
 'OH_035',
 'OH_036',
 'OH_037',
 'OH_038',
 'OH_039',
 'OH_040',
 'OH_041',
 'OH_042',
 'OH_043',
 'OH_044',
 'OH_045',
 'OH_046',
 'OH_047',
 'OH_048',
 'OH_049',
 'OH_050',
 'OH_051',
 'OH_052',
 'OH_053',
 'OH_054',
 'OH_055',
 'OH_056',
 'OH_057',
 'OH_058',
 'OH_059',
 'OH_060',
 'OH_061',
 'OH_062',
 'OH_063',
 'OH_064',
 'OH_065',
 'OH_066',
 'OH_067',
 'OH_068',
 'OH_069',
 'OH_070',
 'OH_071',
 'OH_072',
 'OH_073',
 'OH_074',
 'OH_075',
 'OH_076',
 'OH_077',
 'OH_078',
 'OH_079',
 'OH_080',
 'OH_081',
 'OH_082',
 'OH_083',
 'OH_084',
 'OH_085',
 'OH_086',
 'OH_087',
 'OH_088',
 'OH_089',
 'OH_090',